# Riscrittura codice Cleaning_2 sostituendo l'excel 

Caricamento librerie e file

In [ ]:
from __future__ import annotations
import re
import numpy as np
import pandas as pd

import config
from config import DatasetConfig, ADNIMERGE

ADNIMERGE.source = "ADNIMERGE_cleaned_01.csv"
OUTPUT_FILE = "ADNIMERGE_cleaned_02.csv"

Funzione da richiamare per salvare le modifiche su ADNIMERGE_cleaned_02.csv

In [39]:
def save_dataset(df, path):
   df.to_csv(path, index=False)
   print(f"Salvato: {path}  (shape: {df.shape})")

## Rimozione colonne con troppi valori non validi per l'analizi
Funzione di pulizia colonne. Calcola, per ogni colonna, la percentuale di valori validi e scarta quelle sotto la soglia definita in config.MISSING_KEEP_THRESHOLD.

In [40]:
def remove_param_few_subjects(df, threshold=config.MISSING_KEEP_THRESHOLD):
    df = df.copy()
    valid_ratio = df.notna().mean()
    dropped = valid_ratio[valid_ratio < threshold].index.tolist()
    df = df.drop(columns=dropped)
    return df, dropped

Esecuzione dello step. Legge il file originale, applica la pulizia, stampa quali colonne sono state scartate e la variazione di shape, poi salva il risultato con save_dataset().

In [ ]:
df_raw = pd.read_csv(ADNIMERGE.source)

df_cleaned, dropped_columns = remove_param_few_subjects(df_raw)
 
print(f"[STEP 1] Colonne scartate ({len(dropped_columns)}): {dropped_columns}")
print(f"[STEP 1] Shape: {df_raw.shape} -> {df_cleaned.shape}")
 
save_dataset(df_cleaned, OUTPUT_FILE)  

C:\Users\ifabb\AppData\Local\Temp\ipykernel_23468\2439724107.py:1: DtypeWarning: Columns (0: TTAU_CSF, 1: TAU_bl, 2: PTAU_bl) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(ADNIMERGE.source)


[STEP 1] Colonne scartate (46): ['FDG', 'PIB', 'AV45', 'FBB', 'AB42_CSF', 'TTAU_CSF', 'PT181_CSF', 'DIGITSCOR', 'MOCA', 'EcogPtMem', 'EcogPtLang', 'EcogPtVisspat', 'EcogPtPlan', 'EcogPtOrgan', 'EcogPtDivatt', 'EcogPtTotal', 'EcogSPMem', 'EcogSPLang', 'EcogSPVisspat', 'EcogSPPlan', 'EcogSPOrgan', 'EcogSPDivatt', 'EcogSPTotal', 'FLDSTRENG', 'DIGITSCOR_bl', 'MOCA_bl', 'EcogPtMem_bl', 'EcogPtLang_bl', 'EcogPtVisspat_bl', 'EcogPtPlan_bl', 'EcogPtOrgan_bl', 'EcogPtDivatt_bl', 'EcogPtTotal_bl', 'EcogSPMem_bl', 'EcogSPLang_bl', 'EcogSPVisspat_bl', 'EcogSPPlan_bl', 'EcogSPOrgan_bl', 'EcogSPDivatt_bl', 'EcogSPTotal_bl', 'ABETA_bl', 'TAU_bl', 'PTAU_bl', 'PIB_bl', 'AV45_bl', 'FBB_bl']
[STEP 1] Shape: (11458, 118) -> (11458, 72)


Verifica di coerenza. Rilegge il file appena scritto e controlla che corrisponda davvero a df_cleaned, per evitare di proseguire con un file non aggiornato (il problema riscontrato in precedenza).

In [55]:
check_step1 = pd.read_csv(OUTPUT_FILE)
assert check_step1.shape == df_cleaned.shape, "[STEP 1] File salvato NON corrisponde a df_cleaned!"
print(f"[STEP 1] Verifica OK: '{OUTPUT_FILE}' ha shape {check_step1.shape}")

[STEP 1] Verifica OK: 'ADNIMERGE_cleaned_02.csv' ha shape (11458, 72)


Controlla le dimensioni prima/dopo, per avere conferma numerica:

In [56]:
print("Shape originale:", df.shape)
print("Shape pulito:", df_cleaned.shape)

Shape originale: (11458, 118)
Shape pulito: (11458, 72)


## Conversione in dummy delle colonne 'GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX'
Configurazione dello step. Definisce input/output (il file appena prodotto dallo STEP 1) e l'elenco delle variabili categoriche da convertire.

In [ ]:
DUMMY_INPUT = OUTPUT_FILE             # <-- usa direttamente l'output dello step 1
DUMMY_OUTPUT = "ADNIMERGE_cleaned_02.csv"

REF_LIST = ['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']

Funzione di conversione in dummy. Individua tra ref_list le colonne presenti nel dataframe e le trasforma in variabili binarie con pd.get_dummies.

In [71]:
def classes_to_dummies(df, ref_list):
    df = df.copy()
    to_dummy_list = [c for c in ref_list if c in df.columns]
    if to_dummy_list:
        df = pd.get_dummies(df, columns=to_dummy_list, dtype=int)
    return df, to_dummy_list

Funzione di conteggio dummy. Conta quante colonne dummy sono state create in totale e quante per ciascuna variabile originale.

In [58]:
def count_dummy_columns(original_df, final_df, converted_columns):
    new_dummy_columns = [c for c in final_df.columns if c not in original_df.columns]
    per_column_count = {
        col: len([c for c in new_dummy_columns if c.startswith(col + "_")])
        for col in converted_columns
    }
    return len(new_dummy_columns), per_column_count

Controllo di sicurezza pre-esecuzione. Legge il file d'ingresso e verifica che abbia già il numero di colonne atteso dallo STEP 1 (72), bloccando l'esecuzione con un messaggio chiaro se non è così.

In [59]:
df_step2 = pd.read_csv(DUMMY_INPUT)

In [60]:
assert df_step2.shape[1] == df_cleaned.shape[1], (
    f"[STEP 2] Attenzione: '{DUMMY_INPUT}' ha {df_step2.shape[1]} colonne, "
    f"attese {df_cleaned.shape[1]}. Rieseguire lo STEP 1 prima di procedere."
)

Esecuzione e salvataggio. Applica la conversione in dummy e salva il risultato con save_dataset().

In [72]:
final_df, converted_columns = classes_to_dummies(df_step2, ref_list=REF_LIST)

save_dataset(final_df, DUMMY_OUTPUT)

Salvato: ADNIMERGE_cleaned_02.csv  (shape: (11458, 84))


### Report finale. Riepiloga colonne convertite, variazione di shape e conteggio dettagliato delle dummy create per ciascuna variabile.

In [62]:
total_dummy_count, per_column_count = count_dummy_columns(df_step2, final_df, converted_columns)
print(f"[STEP 2] Colonne convertite in dummy: {converted_columns}")
print(f"[STEP 2] Shape: {df_step2.shape} -> {final_df.shape}")
print(f"[STEP 2] Totale colonne dummy create: {total_dummy_count}")
for col, n in per_column_count.items():
    print(f"  '{col}' -> {n} colonne dummy")

[STEP 2] Colonne convertite in dummy: ['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']
[STEP 2] Shape: (11458, 72) -> (11458, 84)
[STEP 2] Totale colonne dummy create: 17
  'GENDER' -> 2 colonne dummy
  'MARRY' -> 4 colonne dummy
  'ETHNICITY' -> 2 colonne dummy
  'RACE' -> 6 colonne dummy
  'DX' -> 3 colonne dummy


Controllo modifica andata a buon fine.

In [65]:
pd.read_csv("ADNIMERGE_cleaned_02.csv").head(5)

,RID,COLPROT,ORIGPROT,PTID,SITE,VISCODE,EXAMDATE,DX_bl,AGE_bl,EDUCATION,...,ETHNICITY_1.0,RACE_0.0,RACE_1.0,RACE_2.0,RACE_3.0,RACE_4.0,RACE_5.0,DX_0,DX_1,DX_2
0,2,ADNI1,ADNI1,011_S_0002,11,bl,2005-09-08,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0
1,2,ADNI1,ADNI1,011_S_0002,11,m06,2006-03-06,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0
2,2,ADNI1,ADNI1,011_S_0002,11,m36,2008-08-27,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0
3,2,ADNIGO,ADNI1,011_S_0002,11,m60,2010-09-22,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0
4,2,ADNI2,ADNI1,011_S_0002,11,m72,2011-09-19,CN,74.3,16,...,0,0,0,0,0,0,1,1,0,0


Controllo solo le colonne aggiunt dummy.

In [70]:
final_df[[c for c in final_df.columns if c not in df_step2.columns]].tail(5)

,GENDER_0,GENDER_1,MARRY_0.0,MARRY_1.0,MARRY_2.0,MARRY_3.0,ETHNICITY_0.0,ETHNICITY_1.0,RACE_0.0,RACE_1.0,RACE_2.0,RACE_3.0,RACE_4.0,RACE_5.0,DX_0,DX_1,DX_2
11453,0,1,0,1,0,0,0,1,1,0,0,0,0,0,0,1,0
11454,1,0,0,0,1,0,1,0,0,0,0,0,1,0,1,0,0
11455,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1,0
11456,0,1,0,1,0,0,1,0,0,0,0,0,0,1,0,1,0
11457,1,0,0,1,0,0,0,1,1,0,0,0,0,0,1,0,0
